# Staging Layer EDA — `staging` schema
Verifies the clean layer: source-to-staging reconciliation, quarantine
counts, decode distributions, key uniqueness, numeric distributions and
date ranges. Every check here mirrors a staging procedure rule plus a
DQ loop in `tests/data_quality/staging/`.

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg
import seaborn as sns
from dotenv import load_dotenv

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy connectable")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO_ROOT / ".env")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (10, 5)

SRC = os.getenv("POSTGRES_SCHEMA_SOURCE", "source")
STG = "staging"


def pg_conn():
    return psycopg.connect(
        host=os.getenv("POSTGRES_HOST", "localhost"),
        port=os.getenv("POSTGRES_PORT", "5432"),
        dbname=os.getenv("POSTGRES_DB", "datawarehouse"),
        user=os.getenv("POSTGRES_USER", "postgres"),
        password=os.getenv("POSTGRES_PASSWORD", ""),
        autocommit=True,
    )


def read_table(name, schema):
    with pg_conn() as conn:
        return pd.read_sql_query(f'SELECT * FROM "{schema}"."{name}"', conn)


TABLES = ["cust_info", "prd_info", "cust_az12", "loc_a101", "sales_details", "px_cat_g1v2"]
stg = {t: read_table(t, STG) for t in TABLES}
print("staging loaded:", {t: stg[t].shape for t in TABLES})


## 1. Reconciliation — source vs staging row counts
Deltas must be fully explainable: `cust_info` drops 4 null keys and
collapses 6 duplicate groups (18494 → 18484); everything else is 1:1.

In [ ]:
src_names = {"cust_info": "cust_info", "prd_info": "prd_info", "cust_az12": "CUST_AZ12",
             "loc_a101": "LOC_A101", "sales_details": "sales_details", "px_cat_g1v2": "PX_CAT_G1V2"}
rec = pd.DataFrame([{"table": t,
                      "source": read_table(src_names[t], SRC).shape[0],
                      "staging": len(stg[t])} for t in TABLES])
rec["delta"] = rec["staging"] - rec["source"]
rec

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(rec))
w = 0.35
ax.bar([i - w / 2 for i in x], rec["source"], width=w, label="source")
ax.bar([i + w / 2 for i in x], rec["staging"], width=w, label="staging")
ax.set_xticks(list(x), rec["table"], rotation=20)
ax.set_yscale("log")
ax.legend()
ax.set_title("Source vs staging rows (log scale)")
plt.tight_layout()


## 2. Quarantine audit — what the clean rules caught
Counts every rule outcome: `'Unknown'` names, `'n/a'` fallbacks, nulled
birthdates, zeroed costs, nulled dates.

In [ ]:
q = pd.DataFrame([
    {"rule": "null/blank names -> 'Unknown'",
     "count": int(((stg["cust_info"]["cst_first_name"] == "Unknown") | (stg["cust_info"]["cst_last_name"] == "Unknown")).sum())},
    {"rule": "gender -> 'n/a'", "count": int((stg["cust_az12"]["gender"] == "n/a").sum())},
    {"rule": "birthdate nulled (invalid/future)", "count": int(stg["cust_az12"]["birthdate"].isna().sum())},
    {"rule": "cost zeroed", "count": int((stg["prd_info"]["prd_cost"] == 0).sum())},
    {"rule": "null order dates", "count": int(stg["sales_details"]["sls_order_dt"].isna().sum())},
    {"rule": "null ship/due dates",
     "count": int(stg["sales_details"]["sls_ship_dt"].isna().sum() + stg["sales_details"]["sls_due_dt"].isna().sum())},
])
q
q.set_index("rule")["count"].sort_values().plot.barh(figsize=(8, 4), title="Quarantine hits by rule")
plt.tight_layout()


## 3. Decode distributions — gender, marital, line, country
Every value must sit inside its allowed set (see test 09).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
stg["cust_az12"]["gender"].value_counts().plot.bar(ax=axes[0, 0], title="cust_az12.gender")
stg["cust_info"]["cst_marital_status"].value_counts().plot.bar(ax=axes[0, 1], title="marital_status")
stg["prd_info"]["prd_line"].value_counts().plot.bar(ax=axes[1, 0], title="prd_line")
stg["loc_a101"]["cntry"].value_counts().head(10).sort_values().plot.barh(ax=axes[1, 1], title="top countries")
plt.tight_layout()


## 4. Key uniqueness — merge keys must be unique per table.

In [ ]:
uniq = pd.DataFrame([
    {"table": t, "key": k,
     "dup_groups": int((stg[t][k].value_counts() > 1).sum() if isinstance(k, str) else 0)}
    for t, k in [("cust_info", "cst_id"), ("prd_info", "prd_id"), ("cust_az12", "cid"),
                 ("loc_a101", "cid"), ("px_cat_g1v2", "id")]
] + [{"table": "sales_details", "key": "(ord, prd)",
      "dup_groups": int(stg["sales_details"].duplicated(subset=["sls_ord_num", "sls_prd_key"]).sum())}])
uniq


## 5. Numerics — post-clean distributions and boxplots.

In [ ]:
sales = stg["sales_details"].copy()
for c in ["sls_sales", "sls_quantity", "sls_price"]:
    sales[c] = pd.to_numeric(sales[c], errors="coerce")
print(sales[["sls_sales", "sls_quantity", "sls_price"]].describe().T.to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["sls_sales", "sls_quantity", "sls_price"]):
    sns.boxplot(x=sales[col].dropna(), ax=ax)
    ax.set_title(f"{col} boxplot")
plt.tight_layout()


## 6. Dates — load batch, order calendar, ship lag.

In [ ]:
for t in TABLES:
    print(t, "loaded_at distinct:", stg[t]["loaded_at"].nunique(),
          "| min:", stg[t]["loaded_at"].min(), "| max:", stg[t]["loaded_at"].max())

sdt = sales.copy()
for c in ["sls_order_dt", "sls_ship_dt", "sls_due_dt"]:
    sdt[c] = pd.to_datetime(sdt[c], errors="coerce")
sdt["order_month"] = sdt["sls_order_dt"].dt.to_period("M").astype(str)
sdt["order_month"].value_counts().sort_index().plot(figsize=(14, 4), title="Staged orders per month")
plt.tight_layout()

sdt["ship_lag"] = (sdt["sls_ship_dt"] - sdt["sls_order_dt"]).dt.days
print("negative lags:", int((sdt["ship_lag"] < 0).sum()))
sdt["ship_lag"].dropna().plot.hist(bins=30, figsize=(10, 4), title="Ship lag days")
plt.tight_layout()
